# ATLAS *Z*+jets Omnifold: Basic usage and setup

This notebook serves as an introduction to interacting with the ATLAS full-phase space measurement of $Z$+jets production in $pp$ collisions at $\sqrt{s}=13$ TeV.
It is based on the notebooks written for the previous [Multifold $Z$+jets measurement][https://gitlab.cern.ch/atlas-physics/public/sm-z-jets-omnifold-2024].
While the previous measurement targeted 24 observables, the full-phase space measurement is differential in the kinematics of every charged particle, making it a variable dimensional measurement.
For this reason the previously used HDF5 files are replaced with ROOT files and read using an uproot/awkward array interface.
Beyond this the introductory methods shown here are very similar.
Like in the previous round, two other notebooks provide more advanced instructions on how to access and create plots of the differential cross sections, how to obtain the associated covariance, and how to perform statistical compatibility tests: 
- One with results derived from pseudo-data, with a known target: [2_pseudo_results.ipynb](./2_pseudo_results.ipynb). 
- One with the actual measurements based on real data: [3_results.ipynb](./3_results.ipynb).
Finally three notebooks demonstrate advanced use cases of the un-binned spectra like those shown in the publication.

## File structure and content

The following data files are needed to construct the measurement.
All are provided in the ROOT file format:

- Default Monte-Carlo (MadGraph) sample: `ZjetOmnifold_5Jul2025_MGPy8FxFxPlusNonStrong_syst_Test_withdd.root`
- Alternative Monte-Carlo (Sherpa) sample: `ZjetOmnifold_Mar10_Sherpa2211_LookLike_MgFxFx_Test_V5.root`
- Truth Pseudodata, the target of the pseudo-measurement: `TruthPseudodata_Sherpa2211DY_Dibo_EW_PowhegPythiaTop_PosWeights_WithTracks_June2025_shuffled.root`
- Truth MadGraph sample: a set of files describing the truth-level MadGraph predictions
- Truth Sherpa sample: a set of files describing the truth-level Sherpa predictions

**TODO**: Here add information on the file content of the final files we provide for publication. Will need to remove e.g. systematic information from the default MC file

### Data download

To download all of the ROOT files above, run the cell below.
Note this will take a few GB of disk space (**TODO** add exact number)!

In [ ]:
### Download data (reconfigure once data are public)
import os
if os.path.isfile("files/data/multifold.h5"):
    print("Skipping download -- data already exists.")
else:
    !rm -f files.zip
    !wget https://zenodo.org/records/11507450/files/files.zip
    !unzip -o files.zip

try:
    from multifold_util import *
    import numpy as np
    import pandas as pd
    pd.options.mode.chained_assignment = None
    import matplotlib.pyplot as plt
    from tqdm import tqdm
    import os
    import pickle

    if os.path.isfile("files/data/multifold.h5"):
        print("Done!")
    else:
        raise Exception("File does not exist.")

except Exception as e:
    print(f"There was an error: {e}")

Skipping download -- data already exists.
Done!


### Load data

Once the data are downloaded, run the cells below to load it into memory

In [2]:
### Default: load the full measurement
n_events = None

### If running on Binder, which has a memory constraint of 2 GB, uncomment the following line to cut the size of the datasets approximately in half. 
### The results will not be identical to the ones presented in our paper, but can be used for demo & educational purposes.
# n_events = 200_000

In [3]:
### Make sure you have downloaded the public datasets to your local folder, as shown in the README
top_folder = "./files/"

print("Loading the measurement - based on actual data.")
folder = os.path.join(top_folder, "data")

# Nominal measurement with most uncertainties (based on MG5-Drell Yan)
multifold = pd.read_hdf(os.path.join(folder,"multifold.h5"), stop = n_events)
# Sherpa-based, alternative measurement. Difference with nominal is considered as an uncertainty.
multifold_sherpa = pd.read_hdf(os.path.join(folder,"multifold_sherpa.h5"), stop = 0.8*n_events) # the 0.8 is to make the nEff approximately equal to the ratio in the full measurement
# Alternative measurement that includes EW Zjj (i.e. 'VBF') and diboson components in the unfolding. Difference with nominal is considered another uncertainty. 
multifold_nonDY  = pd.read_hdf(os.path.join(folder,"multifold_nonDY.h5"), stop = n_events)

Loading the measurement - based on actual data.


## A look at the event sample content

Assuming the above worked, we can look at a fraction of the key content of `multifold`. This is the most important data file, as it provides the central value and most uncertainties:

In [4]:
multifold

,pT_ll,pT_l1,pT_l2,eta_l1,eta_l2,phi_l1,phi_l2,y_ll,pT_trackj1,y_trackj1,...,weights_trackPtScale,weights_theoryPSjet,weights_theoryPSsoft,weights_theoryMPI,weights_theoryPSscale,weights_theoryAlphaS,weights_theoryQCD,weights_theoryPDF,weights_topBackground,weights_lumi
0,479.442780,288.466919,198.183929,-0.117443,-0.334414,0.893844,0.543039,-0.205878,243.584351,-0.924242,...,0.003174,0.002844,0.003195,0.003290,0.003154,0.003001,0.003243,0.003152,0.003240,0.003423
1,274.524994,166.120789,125.378044,0.313321,0.368928,2.230892,1.537930,0.337239,96.407021,-1.158080,...,0.008168,0.008563,0.008236,0.008147,0.009276,0.008547,0.008121,0.008651,0.008191,0.008061
2,462.713226,335.697479,133.157684,0.766387,0.534180,1.191890,0.832184,0.700256,86.603012,-1.048886,...,0.001638,0.001724,0.001890,0.001724,0.001687,0.001710,0.001437,0.001614,0.001746,0.001806
3,215.157608,189.518021,25.711994,1.083798,-0.190828,2.326127,2.406116,0.904586,222.285919,0.166012,...,0.004669,0.004622,0.004648,0.004564,0.004554,0.004680,0.005080,0.004376,0.004604,0.004761
4,222.458313,128.850159,108.589226,-0.635713,-0.972127,-2.909612,2.656590,-0.789699,206.385544,0.392547,...,0.002102,0.002417,0.002129,0.002297,0.002181,0.002248,0.002304,0.002303,0.002298,0.002259
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
418009,934.971924,738.464722,196.525192,0.102944,-0.133273,2.388935,2.404151,0.053082,475.055756,0.811634,...,0.000069,0.000070,0.000061,0.000073,0.000078,0.000064,0.000060,0.000076,0.000063,0.000068
418010,245.813461,166.847061,93.757919,1.308837,1.533588,1.591105,0.884510,1.389817,713.371887,1.452970,...,0.000193,0.000189,0.000203,0.000198,0.000221,0.000190,0.000216,0.000192,0.000205,0.000197
418011,478.670349,378.737518,108.016479,-0.328871,-0.227935,-1.235470,-1.675783,-0.306456,100.177452,0.050614,...,0.001969,0.001813,0.001825,0.001861,0.001924,0.001852,0.001832,0.001957,0.001893,0.001920
418012,278.586029,249.255356,43.581135,0.632484,0.687496,-0.966683,-0.071259,0.640673,224.825485,1.490560,...,0.003238,0.003101,0.003090,0.003092,0.003129,0.003002,0.002131,0.003295,0.003003,0.003167


We can see that the this event dataset contains 418,014 events, and the dimuon $p_\text{T}$ of the first event is 479 GeV, while the leading and subleading muons have $p_\text{T}$ 288 GeV and 198 GeV, respectively. We can also note that there are 276 "columns", i.e. properties of each event in this dataset file. These include the 24 kinematic observables, and about 250 event weights. Let's investigate this further with the next code box.

In [5]:
def printEventFeatures(dataset,desc):
    columns = np.array(dataset.keys())
    NmcBS=NdataBS=Nsys=Nens=Nvars=0
    for column in columns:
        if "bootstrap_mc" in column:
            NmcBS+=1
        elif "bootstrap_data" in column:
            NdataBS+=1
        elif "weights_ensemble" in column:
            Nens+=1
        elif "weight" in column:
            Nsys+=1
        elif column!="target_dd": # avoid couting special varialbe for DD unfold syst
            Nvars+=1
    print("\nFor {}, there are {} event properties:".format(desc,len(columns)))
    print("  {} variables, {} weights, {} NN ensemble weights, {} MC and {} data bootstrap weights".format(Nvars,Nsys,Nens,NmcBS,NdataBS))

printEventFeatures(multifold,"the main dataset")
printEventFeatures(multifold_sherpa,"the alternative, Sherpa-derived measurement")
printEventFeatures(multifold_nonDY,"the dataset derived with non-Drell-Yan components")


For the main dataset, there are 276 event properties:
  24 variables, 26 weights, 100 NN ensemble weights, 25 MC and 100 data bootstrap weights

For the alternative, Sherpa-derived measurement, there are 51 event properties:
  24 variables, 2 weights, 0 NN ensemble weights, 25 MC and 0 data bootstrap weights

For the dataset derived with non-Drell-Yan components, there are 26 event properties:
  24 variables, 2 weights, 0 NN ensemble weights, 0 MC and 0 data bootstrap weights


From the numbers displayed above, we can see that the three datasets all contain the 24 measured observables. 

There are a series of event weights that provide the measurement: central value + systematic variations. There is also a weight for the MC prediction as further detailed below. 
* The bootstrap weights provide statistical uncertainties. The MC bootstrap weights encodes the statistical uncertainties associated with the MC used to model the detector response (analogous to MC statistical uncertainty on the migration matrix in traditional unfolding). This uncertainty comes from the auxiliary sample used to train the NN models (i.e. it comes from the "training set" used to perform the actual unfolding, while the measurement files released are the "evaluation set").
* The data bootstrap weights encode the statistical uncertainty of the actual data.
* The NN ensemble weights corresponds to the results provided by 100 different neural networks, and their spread is taken as an uncertainty due to the 'NN instability'
* The final statistical uncertainty is the uncertainty on the datasets themselves (i.e. on the evaluation set).


## Obtaining measured cross sections, uncertainties and MC predictions

As described in the paper, the measurement is performed in a fiducial region defined by two opposite charge, prompt muons (prompt meaning that they do not originate from a hadron decay) that each fulfill $p_\mathrm{T}>20$ GeV and $|\eta|<2.5$. The dimuon system is further required to fulfill $m_{\mu\mu} \in (81,101)\,\text{GeV}$ and $p_\text{T}^{\mu\mu}>200\,\text{GeV}$, i.e. be consistent with a boosted $Z$ boson decaying to muons. Summing up `weights_nominal` of all events in the sample will return the central value of the measured cross section of the full fiducial volume. One can further construct a subregion based on any combination of selection criteria using on the 24 event observables (with certain caveats as discussed below) and obtain associated measured (and predicted) cross sections from the sum of weights of the events that fall in this region.

The central value in a subregion $A$ is given by the sum of nominal weights, `weights_nominal`, in the code, which has units femtobarn:

$$
\hat{\sigma}_A = \sum_{i\in A} w^\text{nom}_i.
$$

A systematically varied cross section corresponding to a nuisance parameter $(k)$ can be obtained using any of the alternative such weights:

$$
\hat{\sigma}_A^{(k)} = \sum_{i\in A} w^{(k)}_i.
$$

From such a variation, the absolute uncertainty amplitude would be given by the differnece to the central value: $\Delta_{A}^{(k)} = \hat{\sigma}_{A}^{(k)} - \hat{\sigma}_{A}$, and these variations should be treated as uncorrelated between each other, and fully correlated between bins. As a consequence, the covariance between two regions $A$ and $B$ from a given nuisance parameter $k$ is evaluated as $V_{A,B}^{(k)}=\Delta_{A}^{(k)}\,\Delta_{B}^{(k)}$, and the covariance from several such NPs is given by the sum: $V_{A,B} = \sum_k V_{A,B}^{(k)}$.

Statistical uncertainties are evaluated using bootstrap weights. The statistical covariance between two regions $A$ and $B$ is

$$
V^\text{stat}_{A,B} = \frac{1}{N_\text{BS}}\sum_{b=1}^{N_\mathrm{BS}}(\hat{\sigma}^{(b)}_A-\hat{\sigma}^\text{nom}_A)(\hat{\sigma}^{(b)}_B-\hat{\sigma}^\text{nom}_B).
$$

The code blocks below provide examples of how to extract some of these results. For more detailed checks, such as building the full covariance matrices, see the other two notebooks.

In [6]:
# Method to obtain measured and predicted cross section of an event sample corresponding
# to a (fiducial) kinematic subregion
def printCrossSection(dataset,desc,wnom_name="weights_nominal"):
    meas_xsec = np.sum(dataset[wnom_name])
    pred_xsec = np.sum(dataset.weight_mc)
    # sumw2 = np.sum(dataset[wnom_name]**2)
    # Neff = meas_xsec*meas_xsec/sumw2 => see nEff method
    print("\nFor {}, we have:".format(desc))
    print("  Measured x-sec:     {:.2f} fb".format(meas_xsec))
    print("  MC-predicted x-sec: {:.2f} fb".format(pred_xsec))
    print("  Effective statistics: {:.1f}".format(nEff(dataset[wnom_name])))

printCrossSection(multifold,"nominal, MG5-based result, full fiducial phase space")
printCrossSection(multifold_sherpa,"Sherpa-based result, full fiducial phase space")
printCrossSection(multifold_nonDY,"Non-DY, full fiducial phase space")

# Define a fiducial subset ("A" in equations above)
print("\n\n===== FIDUCIAL VOLUME pT(ll) > 500 GeV =====")

meas_pT500 = multifold[multifold.pT_ll > 500]
sherpa_pT500 = multifold_sherpa[multifold_sherpa.pT_ll > 500]
nonDY_pT500 = multifold_nonDY[multifold_nonDY.pT_ll > 500]

printCrossSection(meas_pT500,"nominal, MG5-based result, in region pT(ll) > 500 GeV")
printCrossSection(sherpa_pT500,"Sherpa-based result, in region pT(ll) > 500 GeV")
printCrossSection(nonDY_pT500,"Non-Drell-Yan result, in region pT(ll) > 500 GeV")



For nominal, MG5-based result, full fiducial phase space, we have:
  Measured x-sec:     1808.34 fb
  MC-predicted x-sec: 1701.13 fb
  Effective statistics: 197876.2

For Sherpa-based result, full fiducial phase space, we have:
  Measured x-sec:     1816.33 fb
  MC-predicted x-sec: 1601.73 fb
  Effective statistics: 179010.5

For Non-DY, full fiducial phase space, we have:
  Measured x-sec:     1809.55 fb
  MC-predicted x-sec: 1780.72 fb
  Effective statistics: 202770.8


===== FIDUCIAL VOLUME pT(ll) > 500 GeV =====

For nominal, MG5-based result, in region pT(ll) > 500 GeV, we have:
  Measured x-sec:     41.18 fb
  MC-predicted x-sec: 39.47 fb
  Effective statistics: 28472.1

For Sherpa-based result, in region pT(ll) > 500 GeV, we have:
  Measured x-sec:     41.98 fb
  MC-predicted x-sec: 34.93 fb
  Effective statistics: 16403.4

For Non-Drell-Yan result, in region pT(ll) > 500 GeV, we have:
  Measured x-sec:     41.82 fb
  MC-predicted x-sec: 43.43 fb
  Effective statistics: 22058.7

The code above shows how to extract the measured central value, and the MC prediction, and effective statistics.

Both the nominal and the Sherpa results are obtained using Drell-Yan MC only. Their MC prediction does hence not include all signal processes, as EW $Zjj$ (aka 'VBF') and $ZV\to Zjj$ is missing. These processes are included in the non-Drell-Yan file.

The effective number of events (effective statistics) is calculated using $N_\text{eff} = \left(\sum_i w_i \right)^{2}/\sum_i w^2_i$. We can note that although there are 418k events in the full sample, after considering their weights, we effectively have only 198k events.

### Uncertainties

The measurement has a total of 29 sources of uncertainty. 

* 22 of these are encoded as alternative weights, and are propagated as standard nuisance parameters.
* Two sources are taken as the difference between the nominal result and results obtained by different MC samples. These are provided as separate datasets (2-point systematics). These account for the so-called hidden-variable uncertainty. 
* One uncertainty accounts for the data-driven unfolding uncertainty.
* Three uncertainties account for stochastic effects and are stored as bootstrap variations: 
    * The data statistical uncertainty
    * The MC statistical uncertainty on the training set
    * The neural network stochastic variation
* Finally, there is a statistical uncertainty of the provided MC events of the dataset.

In [7]:
# Lists of uncertainty variations
syst_nps = ["pileup","lumi","topBackground", # pileup modelling, luminosity, bkg subtraction (top). 3 NPs
            "trackEffMain","trackEffJet","trackFake","trackPtScale", # track systematics. 4 NPs
            "muEffReco","muEffIso","muEffTrack","muEffTrig","muCalID","muCalMS","muCalResBias","muCalScale", # muon eff+calib. 8 NPs
            "theoryPSjet","theoryPSsoft","theoryMPI","theoryPSscale","theoryAlphaS","theoryQCD","theoryPDF"] # theory. 7 NPs

mc_train_stat  = [col for col in multifold.keys() if col.startswith("weights_bootstrap_mc")]
data_stat      = [col for col in multifold.keys() if col.startswith("weights_bootstrap_data")]
NN_stability   = [col for col in multifold.keys() if col.startswith("weights_ensemble")]

def printXsecUnc(dataset,desc,sherpa,nonDY):
    meas_xsec  = np.sum(dataset["weights_nominal"])    # sum w
    Vstat_meas = np.sum(dataset["weights_nominal"]**2) # sum w^2, statisical variance

    Vmeas = 0 # total, fractional variance
    print("\n---------\n{}".format(desc))
    print("   Measured x-sec:   {:.2f} fb".format(meas_xsec))
    print("\n   Systematics:")
    for NP in syst_nps:
        syst_xsec = np.sum(dataset["weights_"+NP])
        unc = (syst_xsec/meas_xsec-1) # fractional uncertainty amplitude
        Vmeas += unc*unc
        print("  {:>15s}:    {:5.2f}%".format(NP,unc*100))
        
    # Measurments performed with alternative MC samples 
    meas_sherpa = np.sum(sherpa["weights_nominal"]) # 'hidden variable uncertainty'
    meas_nonDY  = np.sum(nonDY["weights_nominal"]) # EW Zjj + diboson modelling 
    meas_DD  = np.sum(dataset["weights_dd"]) # Data-driven unfolding uncertainty
    meas_DD_target  = np.sum(dataset["target_dd"])
    print("  {:>15s}:    {:5.2f}%".format("DD unfold. unc.",(meas_DD/meas_DD_target-1)*100))
    
    Vmeas += (meas_DD/meas_DD_target-1)**2
    Vmeas += (meas_sherpa/meas_xsec-1)**2
    Vmeas += (meas_nonDY/meas_xsec-1)**2
    print("  {:>15s}:    {:5.2f}% (HV unfold. unc.)".format("Sherpa meas",(meas_sherpa/meas_xsec-1)*100))
    print("  {:>15s}:    {:5.2f}%".format("Non-DY meas",(meas_nonDY/meas_xsec-1)*100))
    print("  {:>15s}:    {:5.2f}%".format("Total syst",Vmeas**0.5*100))

    print("\n   Stochastic uncertainties:")
    print("  {:>15s}:    {:5.2f}%".format("MC stat",Vstat_meas**0.5/meas_xsec*100))
    Vmeas += Vstat_meas/meas_xsec/meas_xsec
    
    unc = np.std([np.sum(dataset[col]) for col in mc_train_stat])/meas_xsec
    Vmeas += unc*unc
    print("  {:>15s}:    {:5.2f}%".format("MC train stat",unc*100))

    unc = np.std([np.sum(dataset[col]) for col in data_stat])/meas_xsec
    Vmeas += unc*unc
    print("  {:>15s}:    {:5.2f}%".format("Data stat",unc*100))

    unc = np.std([np.sum(dataset[col]) for col in NN_stability])/meas_xsec
    Vmeas += unc*unc
    print("  {:>15s}:    {:5.2f}%".format("NN stability",unc*100))
    print("\n  {:>15s}:    {:5.2f}%".format("Total uncert",Vmeas**0.5*100))

In [8]:
printXsecUnc(multifold,"Full fiducial phase space, i.e. pT(ll) > 200 GeV",multifold_sherpa,multifold_nonDY)


---------
Full fiducial phase space, i.e. pT(ll) > 200 GeV
   Measured x-sec:   1808.34 fb

   Systematics:
           pileup:    -0.10%
             lumi:     1.70%
    topBackground:    -0.20%
     trackEffMain:    -0.04%
      trackEffJet:    -0.02%
        trackFake:    -0.02%
     trackPtScale:    -0.04%
        muEffReco:     0.55%
         muEffIso:     1.15%
       muEffTrack:     0.28%
        muEffTrig:     0.29%
          muCalID:    -0.11%
          muCalMS:    -0.08%
     muCalResBias:     0.02%
       muCalScale:    -0.27%
      theoryPSjet:    -0.01%
     theoryPSsoft:    -0.01%
        theoryMPI:    -0.01%
    theoryPSscale:    -0.08%
     theoryAlphaS:     0.13%
        theoryQCD:     0.03%
        theoryPDF:    -0.01%
  DD unfold. unc.:     0.57%
      Sherpa meas:     0.44% (HV unfold. unc.)
      Non-DY meas:     0.07%
       Total syst:     2.32%

   Stochastic uncertainties:
          MC stat:     0.22%
    MC train stat:     0.09%
        Data stat:     0.23%
  

In [9]:
printXsecUnc(meas_pT500,"pT(ll) > 500 GeV",sherpa_pT500,nonDY_pT500)


---------
pT(ll) > 500 GeV
   Measured x-sec:   41.18 fb

   Systematics:
           pileup:    -0.47%
             lumi:     1.70%
    topBackground:     0.98%
     trackEffMain:     0.88%
      trackEffJet:    -0.83%
        trackFake:    -0.78%
     trackPtScale:    -1.29%
        muEffReco:    -0.47%
         muEffIso:     1.41%
       muEffTrack:    -0.65%
        muEffTrig:     0.92%
          muCalID:     0.16%
          muCalMS:    -0.04%
     muCalResBias:    -0.58%
       muCalScale:    -0.08%
      theoryPSjet:     0.09%
     theoryPSsoft:    -0.08%
        theoryMPI:    -0.72%
    theoryPSscale:     1.53%
     theoryAlphaS:     0.49%
        theoryQCD:     0.18%
        theoryPDF:     0.38%
  DD unfold. unc.:     2.27%
      Sherpa meas:     1.93% (HV unfold. unc.)
      Non-DY meas:     1.53%
       Total syst:     5.12%

   Stochastic uncertainties:
          MC stat:     0.59%
    MC train stat:     0.40%
        Data stat:     1.71%
     NN stability:     6.65%

     T